# GCN Cloud Notebook

This notebook is configured for Google Colab. It mounts Google Drive, locates the repo inside `MyDrive`, installs the graph dependencies, and runs the cleaned lattice stiffness workflow from `colab_gnn_stiffness_prototype.py`.

In [ ]:
import sys

IN_COLAB = 'google.colab' in sys.modules
print(f'Running in Colab: {IN_COLAB}')

if IN_COLAB:
    %pip -q install torch-geometric
else:
    print('Colab dependency install cell skipped.')

In [ ]:
from pathlib import Path
import sys

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    drive_root = Path('/content/drive/MyDrive')
    repo_candidates = [
        drive_root / 'NSF-REU-Summer-26',
        drive_root / 'Colab Notebooks' / 'NSF-REU-Summer-26',
        drive_root / 'Documents' / 'NSF-REU-Summer-26',
    ]
else:
    drive_root = Path.cwd().resolve()
    repo_candidates = [drive_root, *drive_root.parents]

repo_root = next((path.resolve() for path in repo_candidates if path.exists() and (path / 'active_projects').is_dir()), None)
if repo_root is None:
    raise FileNotFoundError(
        'Could not locate the repo root. Move the project into Google Drive and update repo_candidates in this cell.'
    )

pipeline_root = repo_root / 'active_projects' / 'voronoi_lattice_pipeline'
module_dir = pipeline_root / 'gnn_prototype'
if not (module_dir / 'colab_gnn_stiffness_prototype.py').is_file():
    raise FileNotFoundError(f'Module not found at {module_dir}')

if str(module_dir) not in sys.path:
    sys.path.insert(0, str(module_dir))

print(f'Repo root: {repo_root}')
print(f'Pipeline root: {pipeline_root}')

In [ ]:
import pandas as pd
from IPython.display import display

from colab_gnn_stiffness_prototype import (
    TrainingConfig,
    SimpleGNN,
    create_data_loaders,
    default_data_roots,
    evaluate_model,
    load_lattice_dataset,
    normalize_feature_splits,
    plot_prediction_splits,
    plot_training_history,
    predict_on_directory,
    save_run_artifacts,
    set_seed,
    split_dataset,
    summarize_metrics,
    train_model,
)

In [ ]:
config = TrainingConfig()
set_seed(config.seed)

train_root, predict_root = default_data_roots(pipeline_root)
print(f'Train data: {train_root}')
print(f'Prediction data: {predict_root}')
print(f'Device: {config.device}')
print(f'Batch size: {config.batch_size}')
print(f'Hidden dim: {config.hidden_dim}')
print(f'Epochs: {config.total_epochs}')

In [ ]:
dataset = load_lattice_dataset(train_root)
train_data, val_data, test_data = split_dataset(dataset, seed=config.seed)
scaler = normalize_feature_splits(train_data, val_data, test_data)
train_loader, val_loader, test_loader = create_data_loaders(
    train_data,
    val_data,
    test_data,
    batch_size=config.batch_size,
)

model = SimpleGNN(
    input_dim=train_data[0].x.shape[1],
    hidden_dim=config.hidden_dim,
)

print(f'Train samples: {len(train_data)}')
print(f'Validation samples: {len(val_data)}')
print(f'Test samples: {len(test_data)}')
model

In [ ]:
history = train_model(model, train_loader, val_loader, config)
plot_training_history(history, config.epochs_phase1)

In [ ]:
metrics_by_split = {}
split_results = []

for split_name, loader in (("Train", train_loader), ("Validation", val_loader), ("Test", test_loader)):
    predictions, ground_truth, metrics = evaluate_model(model, loader, device=config.device)
    metrics_by_split[split_name] = metrics
    split_results.append((split_name, predictions, ground_truth))

metrics_frame = summarize_metrics(metrics_by_split)
metrics_frame

In [ ]:
plot_prediction_splits(split_results)

In [ ]:
prediction_results, prediction_metrics = predict_on_directory(
    model,
    predict_root,
    scaler,
    device=config.device,
)

prediction_summary = pd.Series(prediction_metrics, name='Prediction Set')
display(prediction_results.head())
display(prediction_summary)

output_dir = pipeline_root / 'gnn_prototype' / 'outputs'
save_run_artifacts(
    output_dir,
    model,
    scaler,
    history,
    metrics_by_split,
    prediction_results=prediction_results,
)
print(f'Saved artifacts to {output_dir}')